In [2]:
import os
import math
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import rustworkx as rx
from scipy.spatial import KDTree
from pretty_warnings import warn
from tqdm import tqdm

In [ ]:
# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================
OSRM_BASE_URL = "http://127.0.0.1:5001"
TRANSFER_PENALTY_SEC = 600
WALK_NEIGHBORS = 201

STOPS_FILE = "data/stops_combined.txt"
STOP_TIMES_FILE = "data/stop_times_combined.txt"
TRIPS_FILE = "data/trips_combined.txt"

SCHOOLS_FILE = "data/good_regular_schools.csv"

OA_GEOJSON_FILE = "data/Output_Areas_Dec_2011_Boundaries_EW_BGC_2022_-9162807391755440229.geojson" 
LSOA_LOOKUP_FILE = "data/OA11_LAD21_LSOA11_MSOA11_LEP21_EN_v3.xlsx"

OUTPUT_CSV = "school_oa_bus_durations.csv"

COMPLETED_SCHOOLS = {}

# ==========================================
# HELPER FUNCTIONS
# ==========================================
def get_walking_durations_batch(source_idx, target_indices, stops_df):
    source_row = stops_df.iloc[source_idx]
    target_rows = stops_df.iloc[target_indices]
    source_coord = f"{source_row['stop_lon']},{source_row['stop_lat']}"
    target_coords = [f"{row['stop_lon']},{row['stop_lat']}" for _, row in target_rows.iterrows()]
    
    all_coords = ";".join([source_coord] + target_coords)
    url = f"{OSRM_BASE_URL}/table/v1/foot/{all_coords}?sources=0"
    try:
        response = requests.get(url).json()
        return response['durations'][0][1:]
    except Exception:
        return [None] * len(target_indices)

def get_walk_durations_from_point(coord, target_indices, stops_df):
    target_rows = stops_df.iloc[target_indices]
    source_coord_str = f"{coord[1]},{coord[0]}"
    target_coords_str = [f"{row['stop_lon']},{row['stop_lat']}" for _, row in target_rows.iterrows()]
    
    all_coords = ";".join([source_coord_str] + target_coords_str)
    url = f"{OSRM_BASE_URL}/table/v1/foot/{all_coords}?sources=0"
    try:
        response = requests.get(url).json()
        return response['durations'][0][1:]
    except Exception:
        warn("No routes found from point")
        return [None] * len(target_indices)

def get_walk_durations_to_point(source_indices, coord, stops_df):
    source_rows = stops_df.iloc[source_indices]
    source_coords_str = [f"{row['stop_lon']},{row['stop_lat']}" for _, row in source_rows.iterrows()]
    target_coord_str = f"{coord[1]},{coord[0]}"
    
    all_coords = ";".join(source_coords_str + [target_coord_str])
    dest_idx = len(source_indices)
    url = f"{OSRM_BASE_URL}/table/v1/foot/{all_coords}?destinations={dest_idx}"
    try:
        response = requests.get(url).json()
        return [row[0] for row in response['durations'][:-1]]
    except Exception:
        warn("No routes found to point")
        return [None] * len(source_indices)

def bus_between(origin, dest, graph, spatial_tree, stops_df, node_to_idx, idx_to_node):
    """Calculates the optimal transit route between two (lat, lon) points using rustworkx A*."""
    ORIGIN_ID, DEST_ID = "VIRTUAL_ORIGIN", "VIRTUAL_DEST"
    
    # Add virtual nodes dynamically
    origin_idx = graph.add_node({"name": ORIGIN_ID, "lat": origin[0], "lon": origin[1]})
    dest_idx = graph.add_node({"name": DEST_ID, "lat": dest[0], "lon": dest[1]})

    # 1. Update Heuristic: It receives the dictionary payload directly
    def manhattan_heuristic(node_data):
        u_stop = node_data["name"][0] if isinstance(node_data["name"], tuple) else node_data["name"]
        
        # If it's the virtual destination, remaining distance is 0
        if u_stop == DEST_ID:
            return 0.0
            
        lat1, lon1 = node_data['lat'], node_data['lon']
        lat2, lon2 = dest[0], dest[1]

        dlat = abs(lat2 - lat1)
        dlon = abs(lon2 - lon1) * math.cos(math.radians(53.8))
        return dlat + dlon

    # 2. Add Goal Function: rustworkx expects a callable to determine the target
    def is_goal(node_data):
        u_stop = node_data["name"][0] if isinstance(node_data["name"], tuple) else node_data["name"]
        return u_stop == DEST_ID

    try:
        _, indices = spatial_tree.query([origin, dest], k=501) 
        o_indices, d_indices = indices[0], indices[1]

        # Origin walking edges
        o_durations = get_walk_durations_from_point(origin, o_indices, stops_df)
        for idx, dur in zip(o_indices, o_durations):
            if dur is not None:
                stop_id = stops_df.iloc[idx]['stop_id']
                if stop_id in node_to_idx:
                    graph.add_edge(origin_idx, node_to_idx[stop_id], {"weight": float(dur), "type": "walk"})
        
        # Destination walking edges
        d_durations = get_walk_durations_to_point(d_indices, dest, stops_df)
        for idx, dur in zip(d_indices, d_durations):
            if dur is not None:
                stop_id = stops_df.iloc[idx]['stop_id']
                if stop_id in node_to_idx:
                    graph.add_edge(node_to_idx[stop_id], dest_idx, {"weight": float(dur), "type": "walk"})

        # 3. Execute A*: Pass the goal_fn instead of dest_idx
        path_indices = rx.astar_shortest_path(
            graph, 
            origin_idx, 
            goal_fn=is_goal, 
            edge_cost_fn=lambda edge: edge["weight"], 
            estimate_cost_fn=manhattan_heuristic
        )
        
        # If path_indices is empty, no path was found
        if not path_indices:
            return None, None

        # Map indices back to your original custom identifier format
        optimal_path = [graph.get_node_data(i)["name"] for i in path_indices]
        
        # Calculate path weight
        optimal_duration = 0.0
        for i in range(len(path_indices) - 1):
            # For multigraphs, if you have multiple edges, get_edge_data might return a list. 
            # If so, you will need to find the min weight. Assuming standard DiGraph behavior here:
            edge_data = graph.get_edge_data(path_indices[i], path_indices[i+1])
            optimal_duration += edge_data["weight"]
            
        return optimal_path, optimal_duration

    except Exception as e:
        # It's good practice to print the exact exception when debugging new backend logic
        # print(f"Routing Error: {e}") 
        return None, None
    finally:
        # Clean up virtual nodes safely using indices
        graph.remove_node(origin_idx)
        graph.remove_node(dest_idx)

In [12]:
# --- 1. Load Bus Data ---
print("Loading BODS transit data...")
stops_df = pd.read_csv(STOPS_FILE)
stop_times_df = pd.read_csv(STOP_TIMES_FILE)
trips = pd.read_csv(TRIPS_FILE)

stop_times_df["arrival_time"] = pd.to_timedelta(stop_times_df["arrival_time"])
stop_times_df["departure_time"] = pd.to_timedelta(stop_times_df["departure_time"])

stop_times_df = stop_times_df.merge(trips[["route_id", "trip_id"]], on='trip_id', how='left')
stops_loc_dict = stops_df.set_index("stop_id")[["stop_lat", "stop_lon"]].to_dict("index")

grouped = stop_times_df.groupby("trip_id")["stop_sequence"]
stop_times_df["trip_len"] = (grouped.transform("max") - grouped.transform("min"))
trip_stops_dict = stop_times_df.groupby('trip_id')['stop_id'].apply(set).to_dict()

trip_info = stop_times_df[['route_id', 'trip_id', 'stop_sequence', 'trip_len']].drop_duplicates()
trip_info = trip_info.sort_values(by=['route_id', 'trip_len'], ascending=[True, False])

print("Filtering longest unique trips...")
include_ids = []
considered_routes = set()

for route_id, route_trips in tqdm(trip_info.groupby('route_id')):
    if route_id in considered_routes or route_trips.empty:
        continue
    considered_routes.add(route_id)
    
    longest_trip_id = route_trips.iloc[0]['trip_id']
    include_ids.append(longest_trip_id)
    longest_trip_stops = list(trip_stops_dict[longest_trip_id])
    
    if len(longest_trip_stops) >= 2:
        sample_stops = set(np.random.choice(longest_trip_stops, 2, replace=False))
        for trip_id in route_trips['trip_id'].iloc[1:]:
            if trip_stops_dict[trip_id].isdisjoint(sample_stops):
                include_ids.append(trip_id)
                break

longest_trips_df = stop_times_df[stop_times_df["trip_id"].isin(include_ids)].reset_index(drop=True)

Loading BODS transit data...
Filtering longest unique trips...


100%|██████████| 1214/1214 [00:01<00:00, 1194.00it/s]


In [13]:
# --- 2. Calculate Average Segment Times ---
print("Calculating segment travel times...")
df = stop_times_df.sort_values(by=["trip_id", "stop_sequence"])
df["time_from_prev"] = df["arrival_time"] - df.groupby("trip_id")["departure_time"].shift(1)
df["prev_stop_id"] = df.groupby("trip_id")["stop_id"].shift(1)
df["travel_time_seconds"] = df["time_from_prev"].dt.total_seconds()

valid_segments = df.dropna(subset=["prev_stop_id", "travel_time_seconds"])
average_segment_times = valid_segments.groupby(["route_id", "prev_stop_id", "stop_id"])["travel_time_seconds"].mean().reset_index()
mean_times_dict = average_segment_times.set_index(["route_id", "prev_stop_id", "stop_id"])["travel_time_seconds"].to_dict()

Calculating segment travel times...


In [14]:
# --- 3. Build the rustworkx Graph ---
print("Constructing Directed Graph via rustworkx...")
G = rx.PyDiGraph()
node_to_idx = {}
idx_to_node = {}

def get_or_create_node(node_key):
    if node_key not in node_to_idx:
        stop_id_key = node_key[0] if isinstance(node_key, tuple) else node_key
        lat = stops_loc_dict[stop_id_key]["stop_lat"] if stop_id_key in stops_loc_dict else 0.0
        lon = stops_loc_dict[stop_id_key]["stop_lon"] if stop_id_key in stops_loc_dict else 0.0
        
        idx = G.add_node({"name": node_key, "lat": lat, "lon": lon})
        node_to_idx[node_key] = idx
        idx_to_node[idx] = node_key
    return node_to_idx[node_key]

longest_trips_df = longest_trips_df.sort_values(["trip_id", "stop_sequence"])

for trip_id, trip_to_plot in longest_trips_df.groupby("trip_id"):
    trip_to_plot = trip_to_plot.reset_index(drop=True)
    route_id = trip_to_plot["route_id"].iloc[0]

    for i in range(len(trip_to_plot) - 1):
        cur_stop = trip_to_plot.loc[i, "stop_id"]
        next_stop = trip_to_plot.loc[i+1, "stop_id"]

        cur_platform = (cur_stop, route_id)
        next_platform = (next_stop, route_id)

        lookup_key = (route_id, cur_stop, next_stop)
        if lookup_key in mean_times_dict:
            duration_sec = mean_times_dict[lookup_key]
        else:
            duration = trip_to_plot.loc[i+1, "arrival_time"] - trip_to_plot.loc[i, "departure_time"]
            duration_sec = duration.total_seconds()
        
        if duration_sec < 0 or pd.isna(duration_sec):
            continue

        u = get_or_create_node(cur_platform)
        v = get_or_create_node(next_platform)
        G.add_edge(u, v, {"weight": float(duration_sec), "type": "transit"})
        
        # Transfer handling
        c_stop_idx = get_or_create_node(cur_stop)
        n_stop_idx = get_or_create_node(next_stop)
        
        if not G.has_edge(c_stop_idx, u):
            G.add_edge(c_stop_idx, u, {"weight": float(TRANSFER_PENALTY_SEC), "type": "transfer"})
        if not G.has_edge(n_stop_idx, v):
            G.add_edge(n_stop_idx, v, {"weight": float(TRANSFER_PENALTY_SEC), "type": "transfer"})
            
        if not G.has_edge(u, c_stop_idx):
            G.add_edge(u, c_stop_idx, {"weight": 0.0, "type": "exit"})
        if not G.has_edge(v, n_stop_idx):
            G.add_edge(v, n_stop_idx, {"weight": 0.0, "type": "exit"})

Constructing Directed Graph via rustworkx...


In [15]:
# --- 4. Walking OSRM Transfers & KDTree ---
print("Calculating and adding walking transfers...")
coordinates = stops_df[['stop_lat', 'stop_lon']].values
spatial_tree = KDTree(coordinates)

for idx, row in tqdm(stops_df.iterrows(), total=len(stops_df)):
    current_stop_id = row['stop_id']
    if current_stop_id not in node_to_idx:
        continue
    u_idx = node_to_idx[current_stop_id]
    
    _, indices = spatial_tree.query([row['stop_lat'], row['stop_lon']], k=WALK_NEIGHBORS)
    target_indices = indices[1:]
    
    durations = get_walking_durations_batch(idx, target_indices, stops_df)
    
    for target_idx, duration_sec in zip(target_indices, durations):
        if duration_sec is not None:
            neighbor_stop_id = stops_df.iloc[target_idx]['stop_id']
            if neighbor_stop_id in node_to_idx:
                v_idx = node_to_idx[neighbor_stop_id]
                G.add_edge(u_idx, v_idx, {"weight": float(duration_sec), "type": "walk"})
                G.add_edge(v_idx, u_idx, {"weight": float(duration_sec), "type": "walk"})

Calculating and adding walking transfers...


100%|██████████| 32550/32550 [15:19<00:00, 35.41it/s]


In [ ]:
# --- 5. Prepare Geospatial Data ---
print("Preparing OA and School mapping data...")
school_data = pd.read_csv(SCHOOLS_FILE)
oa21lsoa = pd.read_excel(LSOA_LOOKUP_FILE)
oa_gdf = gpd.read_file(OA_GEOJSON_FILE)

selected_oas = oa21lsoa.loc[oa21lsoa["LAD21NM"].isin(["Bradford", "Leeds", "Calderdale", "Kirklees", "Craven"]), "OA11CD"].unique()
bradford_gdf = oa_gdf.loc[oa_gdf["OA11CD"].isin(selected_oas)]

oa_unique = bradford_gdf.drop_duplicates(subset=["OA11CD"])
centroids_4326 = oa_unique.to_crs(epsg=27700).geometry.centroid.to_crs(epsg=4326)
oa_centroid_dict = dict(zip(oa_unique["OA11CD"], centroids_4326))

Preparing OA and School mapping data...


In [17]:
# --- 6. Batch Routing & Save to CSV ---
file_exists = os.path.exists(OUTPUT_CSV)

for index, school in school_data.iterrows():
    school_name = school["Name"]
    if school_name in COMPLETED_SCHOOLS:
        print(f"[{school_name}] already completed, skipping.")
        continue

    school_lat = school["Lat"]
    school_lon = school["Lon"]
    
    print(f"\nCalculating routes for: {school_name}...")
    school_records = []

    for oa in tqdm(bradford_oas):
        oa_centroid = oa_centroid_dict.get(oa)
        if not oa_centroid:
            continue
            
        optimal_path, optimal_duration = bus_between(
            (school_lat, school_lon), 
            (oa_centroid.y, oa_centroid.x),
            G, spatial_tree, stops_df, node_to_idx, idx_to_node
        )
        
        school_records.append({
            "School_Name": school_name,
            "School_Lat": school_lat,
            "School_Lon": school_lon,
            "OA_Code": oa,
            "OA_Centroid_Y": oa_centroid.y,
            "OA_Centroid_X": oa_centroid.x,
            "Duration_Seconds": optimal_duration,
            "Duration_Minutes": optimal_duration // 60 if optimal_duration else None,
            "Path": optimal_path
        })
    
    school_df = pd.DataFrame(school_records)
    school_df.to_csv(OUTPUT_CSV, mode='a', index=False, header=not file_exists)
    file_exists = True 
    
    print(f"Successfully saved data for {school_name} to {OUTPUT_CSV}")


school_df

[Hazelbeck Special School] already completed, skipping.
[Dixons Trinity Academy] already completed, skipping.
[Otley Prince Henry's Grammar School Specialist Language College] already completed, skipping.
[Carlton Bolling] already completed, skipping.
[Bradford Academy] already completed, skipping.
[Co-op Academy Grange] already completed, skipping.
[BBG Academy] already completed, skipping.
[Woodhouse Grove School] already completed, skipping.
[Fulneck School] already completed, skipping.
[Crawshaw Academy] already completed, skipping.
[Pudsey Grammar School] already completed, skipping.
[Titus Salt School] already completed, skipping.
[Beckfoot School] already completed, skipping.
[Jaamiatul Imaam Muhammad Zakaria] already completed, skipping.
[St. Mary's Menston, a Catholic Voluntary Academy] already completed, skipping.
[South Craven School] already completed, skipping.
[West Specialist Inclusive Learning Centre (Mainsite)] already completed, skipping.
[The Crossley Heath School] a

100%|██████████| 1528/1528 [19:08<00:00,  1.33it/s]


Successfully saved data for Hanson Academy to school_oa_travel_durations.csv

Calculating routes for: Appleton Academy...


100%|██████████| 1528/1528 [35:22<00:00,  1.39s/it] 


Successfully saved data for Appleton Academy to school_oa_travel_durations.csv

Calculating routes for: Dixons City Academy...


...

100%|██████████| 1528/1528 [3:21:43<00:00,  7.92s/it]     


Successfully saved data for Bronte Girls' Secondary Academy to school_oa_travel_durations.csv

Calculating routes for: Dixons McMillan Academy...


100%|██████████| 1528/1528 [30:28<00:00,  1.20s/it] 

Successfully saved data for Dixons McMillan Academy to school_oa_travel_durations.csv


,School_Name,School_Lat,School_Lon,OA_Code,OA_Centroid_Y,OA_Centroid_X,Duration_Seconds,Duration_Minutes,Path
0,Dixons McMillan Academy,53.786417,-1.765239,E00053353,53.859310,-1.775570,3750.200000,62.0,"[VIRTUAL_ORIGIN, 450023311, (450023311, 14180)..."
1,Dixons McMillan Academy,53.786417,-1.765239,E00053354,53.853975,-1.776242,3576.300000,59.0,"[VIRTUAL_ORIGIN, 450023311, (450023311, 14180)..."
2,Dixons McMillan Academy,53.786417,-1.765239,E00053355,53.852518,-1.755849,3257.200000,54.0,"[VIRTUAL_ORIGIN, 450023311, (450023311, 12780)..."
3,Dixons McMillan Academy,53.786417,-1.765239,E00053356,53.850705,-1.750538,3032.700000,50.0,"[VIRTUAL_ORIGIN, 450023311, (450023311, 12780)..."
4,Dixons McMillan Academy,53.786417,-1.765239,E00053357,53.847899,-1.776531,3380.700000,56.0,"[VIRTUAL_ORIGIN, 450023311, (450023311, 14180)..."
...,...,...,...,...,...,...,...,...,...
1523,Dixons McMillan Academy,53.786417,-1.765239,E00176211,53.809594,-1.775727,2025.100000,33.0,"[VIRTUAL_ORIGIN, 450032476, (450032476, 12766)..."
1524,Dixons McMillan Academy,53.786417,-1.765239,E00176212,53.851534,-1.817638,4454.066667,74.0,"[VIRTUAL_ORIGIN, 450023311, (450023311, 22584)..."
1525,Dixons McMillan Academy,53.786417,-1.765239,E00176213,53.843245,-1.837580,3528.239216,58.0,"[VIRTUAL_ORIGIN, 450023311, (450023311, 22584)..."
1526,Dixons McMillan Academy,53.786417,-1.765239,E00176214,53.828050,-1.734213,2797.634234,46.0,"[VIRTUAL_ORIGIN, 450022310, (450022310, 12757)..."
